# Taller 02 — Aprendizaje Supervisado
## Regresión: Predicción de Costos de Seguro Médico

**Dataset:** Medical Cost Insurance (1,338 registros)  
**Target:** `charges` (costo médico en USD)  
**Autores:** Javier Daza Olivella · Pablo Jimeno Juca · María Sofía Uribe  

## 1. Importación de Librerías
Cargamos todas las librerías necesarias para el análisis, modelado y visualización.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Carga de Datos
Cargamos el dataset desde su URL pública. Se trata del clásico dataset de costos de seguro médico con variables demográficas y de estilo de vida.

In [2]:
from pathlib import Path

possible_paths = [Path('../data/raw/insurance.csv'), Path('data/raw/insurance.csv')]
DATA_PATH = next((path for path in possible_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('No se encontro data/raw/insurance.csv')

df = pd.read_csv(DATA_PATH)
print(f'Ruta usada: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape}')
df.head()


Ruta usada: /Users/mariasofiauribe/Projects/02_Active/intro_ia_2026/Workshop_02/data/raw/insurance.csv
Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
print("Tipos de datos:")
print(df.dtypes)
print(f"\nValores faltantes:\n{df.isnull().sum()}")
print(f"\nEstadísticos descriptivos:")
df.describe()

Tipos de datos:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

Valores faltantes:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Estadísticos descriptivos:


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Distribución de la variable objetivo
Examinamos la distribución de `charges` para detectar asimetría y posibles transformaciones necesarias.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['charges'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de charges')
axes[0].set_xlabel('Costo (USD)')
axes[1].hist(np.log1p(df['charges']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribución de log(charges)')
axes[1].set_xlabel('log(1 + Costo)')
plt.tight_layout()
plt.show()
print(f"Asimetría (skewness): {df['charges'].skew():.3f}")
print(f"Curtosis: {df['charges'].kurtosis():.3f}")

Asimetría (skewness): 1.516
Curtosis: 1.606


**Conclusión:** La distribución de `charges` presenta una fuerte asimetría positiva (skewness > 1), impulsada por el subgrupo de fumadores con costos muy elevados. No se aplicará transformación logarítmica para mantener la interpretabilidad del modelo, pero se tendrá en cuenta al evaluar los residuos.

### 3.2 Correlación entre variables

In [5]:
df_enc = df.copy()
df_enc['sex'] = (df_enc['sex'] == 'male').astype(int)
df_enc['smoker'] = (df_enc['smoker'] == 'yes').astype(int)
df_enc = pd.get_dummies(df_enc, columns=['region'], drop_first=True)
corr = df_enc.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

**Conclusión:** `smoker` tiene la correlación más alta con `charges` (~0.79), lo que lo convierte en el predictor más importante. `age` (~0.30) y `bmi` (~0.20) también contribuyen significativamente. Las variables de región y sexo tienen correlaciones débiles.

### 3.3 Análisis de variables categóricas

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['sex', 'smoker', 'region']):
    df.groupby(col)['charges'].mean().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Costo promedio por {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Charges (USD)')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

**Conclusión:** Los fumadores tienen costos promedio 4x mayores que los no fumadores. La región y el sexo tienen un impacto mucho menor. Este hallazgo confirma que `smoker` será la feature más importante en todos los modelos.

### 3.4 Relación BMI vs Charges

In [7]:
plt.figure(figsize=(10, 5))
colors = df['smoker'].map({'yes': 'red', 'no': 'steelblue'})
plt.scatter(df['bmi'], df['charges'], c=colors, alpha=0.5, s=30)
plt.xlabel('BMI')
plt.ylabel('Charges (USD)')
plt.title('BMI vs Charges (rojo=fumador, azul=no fumador)')
plt.tight_layout()
plt.show()

**Conclusión:** Se observan dos nubes de puntos claramente separadas: fumadores con BMI alto tienen costos extremadamente altos (posible interacción BMI × smoker). Esta interacción no-lineal favorece a los modelos de ensamble sobre la regresión lineal.

## 4. Preprocesamiento

Aplicamos encoding a las variables categóricas. El escalamiento se manejará dentro de los Pipelines para evitar *data leakage*.

In [8]:
INSURANCE_REGION_LEVELS = ['northeast', 'northwest', 'southeast', 'southwest']


def preprocess(df):
    df = df.copy()
    df['age'] = pd.to_numeric(df['age'], errors='raise')
    df['bmi'] = pd.to_numeric(df['bmi'], errors='raise')
    df['children'] = pd.to_numeric(df['children'], errors='raise')
    df['sex'] = df['sex'].astype(str).str.strip().str.lower().map({'female': 0, 'male': 1})
    df['smoker'] = df['smoker'].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})

    region = pd.Categorical(
        df['region'].astype(str).str.strip().str.lower(),
        categories=INSURANCE_REGION_LEVELS,
    )
    region_dummies = pd.get_dummies(region, prefix='region', dtype=int)
    region_dummies = region_dummies.reindex(
        columns=[f'region_{name}' for name in INSURANCE_REGION_LEVELS],
        fill_value=0,
    ).drop(columns=['region_northeast'])

    X = pd.concat([
        df[['age', 'sex', 'bmi', 'children', 'smoker']].astype(float),
        region_dummies,
    ], axis=1)
    y = pd.to_numeric(df['charges'], errors='raise')
    return X, y


X, y = preprocess(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Features: {list(X.columns)}')


Train: (1070, 8), Test: (268, 8)
Features: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']


**Nota sobre Data Leakage:** El `StandardScaler` se ajusta (`fit`) **únicamente** sobre `X_train` a través del Pipeline de sklearn. Si se ajustara sobre el dataset completo, la media y desviación estándar del scaler contendrían información del test set, contaminando la evaluación.

## 5. Ingeniería de Características

Usamos un Random Forest preliminar para identificar las features más importantes antes del entrenamiento final.

In [9]:
from sklearn.ensemble import RandomForestRegressor
rf_prelim = RandomForestRegressor(n_estimators=100, random_state=42)
rf_prelim.fit(X_train, y_train)
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': rf_prelim.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=True)
plt.figure(figsize=(8, 5))
plt.barh(fi_df['Feature'], fi_df['Importance'], color='steelblue')
plt.xlabel('Importancia')
plt.title('Importancia de Features (RF Preliminar)')
plt.tight_layout()
plt.show()

**Conclusión:** `smoker` domina con >50% de importancia total. `age` y `bmi` son los siguientes predictores más relevantes. Las variables de región contribuyen mínimamente. Se mantienen todas las features para el entrenamiento final, aprovechando que los modelos de ensamble manejan eficientemente la multicolinealidad.

## 6. Entrenamiento y Evaluación de Modelos

Entrenamos 5 modelos con `KFold(5)` y `GridSearchCV` para optimizar hiperparámetros. Evaluamos R², MAE y RMSE en el conjunto de prueba.

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# Linear Regression
pipe_lr = Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])
cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=kf, scoring='r2')
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
results['Linear Regression'] = {
    'cv_r2': cv_lr.mean(), 'cv_r2_std': cv_lr.std(),
    'r2': r2_score(y_test, y_pred_lr),
    'mae': mean_absolute_error(y_test, y_pred_lr),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
}
print("Linear Regression: done")

# Ridge
gs_ridge = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('m', Ridge())]),
    {'m__alpha': [0.1, 1, 10, 100]}, cv=kf, scoring='r2', n_jobs=1)
gs_ridge.fit(X_train, y_train)
y_pred_ridge = gs_ridge.predict(X_test)
cv_ridge = cross_val_score(gs_ridge.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results['Ridge'] = {
    'cv_r2': cv_ridge.mean(), 'cv_r2_std': cv_ridge.std(),
    'r2': r2_score(y_test, y_pred_ridge),
    'mae': mean_absolute_error(y_test, y_pred_ridge),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    'best_params': gs_ridge.best_params_,
}
print(f"Ridge: done — best alpha={gs_ridge.best_params_}")

# Lasso
gs_lasso = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('m', Lasso(max_iter=10000))]),
    {'m__alpha': [0.1, 1, 10, 100]}, cv=kf, scoring='r2', n_jobs=1)
gs_lasso.fit(X_train, y_train)
y_pred_lasso = gs_lasso.predict(X_test)
cv_lasso = cross_val_score(gs_lasso.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results['Lasso'] = {
    'cv_r2': cv_lasso.mean(), 'cv_r2_std': cv_lasso.std(),
    'r2': r2_score(y_test, y_pred_lasso),
    'mae': mean_absolute_error(y_test, y_pred_lasso),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
    'best_params': gs_lasso.best_params_,
}
print(f"Lasso: done — best alpha={gs_lasso.best_params_}")

# Random Forest
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
    cv=kf, scoring='r2', n_jobs=1)
gs_rf.fit(X_train, y_train)
y_pred_rf = gs_rf.predict(X_test)
cv_rf = cross_val_score(gs_rf.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results['Random Forest'] = {
    'cv_r2': cv_rf.mean(), 'cv_r2_std': cv_rf.std(),
    'r2': r2_score(y_test, y_pred_rf),
    'mae': mean_absolute_error(y_test, y_pred_rf),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'best_params': gs_rf.best_params_,
    'model': gs_rf.best_estimator_,
}
print(f"Random Forest: done — best={gs_rf.best_params_}")

# Gradient Boosting
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]},
    cv=kf, scoring='r2', n_jobs=1)
gs_gb.fit(X_train, y_train)
y_pred_gb = gs_gb.predict(X_test)
cv_gb = cross_val_score(gs_gb.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results['Gradient Boosting'] = {
    'cv_r2': cv_gb.mean(), 'cv_r2_std': cv_gb.std(),
    'r2': r2_score(y_test, y_pred_gb),
    'mae': mean_absolute_error(y_test, y_pred_gb),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    'best_params': gs_gb.best_params_,
    'model': gs_gb.best_estimator_,
}
print(f"Gradient Boosting: done — best={gs_gb.best_params_}")

Linear Regression: done


Ridge: done — best alpha={'m__alpha': 1}
Lasso: done — best alpha={'m__alpha': 100}


Random Forest: done — best={'max_depth': 10, 'n_estimators': 100}


Gradient Boosting: done — best={'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}


## 7. Comparación de Resultados

In [11]:
df_res = pd.DataFrame([
    {'Modelo': k, 'CV R² (mean)': f"{v['cv_r2']:.4f}", 'CV R² (std)': f"±{v['cv_r2_std']:.4f}",
     'Test R²': f"{v['r2']:.4f}", 'Test MAE': f"${v['mae']:,.0f}", 'Test RMSE': f"${v['rmse']:,.0f}"}
    for k, v in results.items()
])
print(df_res.to_string(index=False))

           Modelo CV R² (mean) CV R² (std) Test R² Test MAE Test RMSE
Linear Regression       0.7389     ±0.0311  0.7836   $4,181    $5,796
            Ridge       0.7389     ±0.0311  0.7835   $4,183    $5,797
            Lasso       0.7393     ±0.0297  0.7806   $4,211    $5,836
    Random Forest       0.8329     ±0.0252  0.8656   $2,537    $4,569
Gradient Boosting       0.8563     ±0.0288  0.8812   $2,473    $4,294


In [12]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
names = list(results.keys())
r2_vals   = [results[n]['r2'] for n in names]
mae_vals  = [results[n]['mae'] for n in names]
rmse_vals = [results[n]['rmse'] for n in names]
colors = ['#3a6ea5', '#e07b39', '#2e9e6a', '#c0392b', '#8e44ad']

for ax, vals, title in zip(axes, [r2_vals, mae_vals, rmse_vals], ['Test R²', 'Test MAE ($)', 'Test RMSE ($)']):
    ax.bar(names, vals, color=colors, edgecolor='white')
    ax.set_title(title)
    ax.set_xticklabels(names, rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [13]:
# Actual vs Predicted — best model
best_name = max(results, key=lambda k: results[k]['r2'])
best_preds = {'Linear Regression': y_pred_lr, 'Ridge': y_pred_ridge, 'Lasso': y_pred_lasso,
              'Random Forest': y_pred_rf, 'Gradient Boosting': y_pred_gb}[best_name]
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_preds, alpha=0.4, color='steelblue', s=25)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Ideal')
plt.xlabel('Valor Real (USD)')
plt.ylabel('Predicción (USD)')
plt.title(f'Real vs Predicho — {best_name}')
plt.legend()
plt.tight_layout()
plt.show()

**Conclusiones finales:**

1. **Mejor modelo:** Los modelos de ensamble (Random Forest y Gradient Boosting) superan claramente a los lineales gracias a su capacidad de capturar la interacción no-lineal `bmi × smoker`.

2. **Modelos lineales:** Ridge y Lasso obtienen resultados similares a OLS (la regularización tiene poco efecto dado el bajo número de features), pero Lasso puede llevar a cero el coeficiente de features poco informativas.

3. **Overfitting:** La diferencia entre CV R² y Test R² es pequeña en todos los modelos, indicando buena generalización.

4. **Recomendación de despliegue:** Gradient Boosting como modelo de producción, con monitoreo periódico del drift en la distribución de `smoker`.

5. **Limitación:** El dataset de 1,338 muestras es pequeño. Con más datos, un modelo de Random Forest con mayor profundidad podría capturar aún mejor la heterogeneidad del costo médico.